In [1]:
   !pip install scikit-learn==1.6.1 scikeras -q

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
# Load Data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=cols)
print("Shape:", df.shape)
df.head()

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# Explore
df.info()
print(df.describe())
print(df['Outcome'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.

In [5]:
# preprocessing
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)
df[zero_cols] = df[zero_cols].fillna(df[zero_cols].median())
print("Missing values left:", df.isnull().sum().sum())

X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(X_train.shape, X_test.shape)

Missing values left: 0
(614, 8) (154, 8)


In [9]:
# Our own wrapper so that a Keras model can be used inside GridSearchCV
class KerasWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, batch_size=10, epochs=50, learning_rate=0.001,
                 init_mode='uniform', activation='relu',
                 dropout_rate=0.0, neurons=12):
        self.batch_size = batch_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.init_mode = init_mode
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.neurons = neurons

    def fit(self, X, y):
        keras.backend.clear_session()
        self.classes_ = np.unique(y)
        self.model_ = keras.Sequential([
            layers.Input(shape=(X.shape[1],)),
            layers.Dense(self.neurons, kernel_initializer=self.init_mode,
                         activation=self.activation),
            layers.Dropout(self.dropout_rate),
            layers.Dense(1, kernel_initializer=self.init_mode, activation='sigmoid')
        ])
        self.model_.compile(loss='binary_crossentropy',
                            optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate),
                            metrics=['accuracy'])
        self.model_.fit(X, y, batch_size=self.batch_size,
                        epochs=self.epochs, verbose=0)
        return self

    def predict(self, X):
        probs = self.model_.predict(X, verbose=0)
        return (probs > 0.5).astype(int).ravel()


model = KerasWrapper()


def run_grid(model, param_grid, name):
    grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=1)
    result = grid.fit(X_train, y_train)
    print(f"--- {name} ---")
    print("Best: %.4f using %s" % (result.best_score_, result.best_params_))
    for m, s, p in zip(result.cv_results_['mean_test_score'],
                       result.cv_results_['std_test_score'],
                       result.cv_results_['params']):
        print("%.4f (%.4f) with: %r" % (m, s, p))
    return result

In [10]:
# batch size and epochs
res_batch = run_grid(model, {'batch_size': [10, 20, 40, 60, 80, 100],
                             'epochs': [10, 50, 100]}, "Batch size and epochs")
best_batch = res_batch.best_params_['batch_size']
best_epochs = res_batch.best_params_['epochs']
model.set_params(batch_size=best_batch, epochs=best_epochs)

--- Batch size and epochs ---
Best: 0.7752 using {'batch_size': 60, 'epochs': 50}
0.7492 (0.0106) with: {'batch_size': 10, 'epochs': 10}
0.7524 (0.0233) with: {'batch_size': 10, 'epochs': 50}
0.7541 (0.0179) with: {'batch_size': 10, 'epochs': 100}
0.7508 (0.0085) with: {'batch_size': 20, 'epochs': 10}
0.7459 (0.0210) with: {'batch_size': 20, 'epochs': 50}
0.7492 (0.0222) with: {'batch_size': 20, 'epochs': 100}
0.7557 (0.0141) with: {'batch_size': 40, 'epochs': 10}
0.7557 (0.0176) with: {'batch_size': 40, 'epochs': 50}
0.7573 (0.0199) with: {'batch_size': 40, 'epochs': 100}
0.7280 (0.0135) with: {'batch_size': 60, 'epochs': 10}
0.7752 (0.0159) with: {'batch_size': 60, 'epochs': 50}
0.7524 (0.0199) with: {'batch_size': 60, 'epochs': 100}
0.7280 (0.0081) with: {'batch_size': 80, 'epochs': 10}
0.7589 (0.0172) with: {'batch_size': 80, 'epochs': 50}
0.7492 (0.0067) with: {'batch_size': 80, 'epochs': 100}
0.7150 (0.0182) with: {'batch_size': 100, 'epochs': 10}
0.7573 (0.0121) with: {'batch_si

KerasWrapper(batch_size=60)

In [11]:
# learning rate:
res_lr = run_grid(model, {'learning_rate': [0.0001, 0.001, 0.01, 0.1, 0.2]}, "Learning rate")
best_lr = res_lr.best_params_['learning_rate']
model.set_params(learning_rate=best_lr)

--- Learning rate ---
Best: 0.7622 using {'learning_rate': 0.001}
0.7264 (0.0145) with: {'learning_rate': 0.0001}
0.7622 (0.0062) with: {'learning_rate': 0.001}
0.7573 (0.0079) with: {'learning_rate': 0.01}
0.7313 (0.0145) with: {'learning_rate': 0.1}
0.7590 (0.0021) with: {'learning_rate': 0.2}


KerasWrapper(batch_size=60)

In [12]:
# weight initialization
res_init = run_grid(model, {'init_mode': ['uniform', 'lecun_uniform', 'normal', 'zeros',
                    'glorot_normal', 'glorot_uniform', 'he_normal', 'he_uniform']},
                    "Weight initialization")
best_init = res_init.best_params_['init_mode']
model.set_params(init_mode=best_init)

--- Weight initialization ---
Best: 0.7606 using {'init_mode': 'uniform'}
0.7606 (0.0159) with: {'init_mode': 'uniform'}
0.7557 (0.0035) with: {'init_mode': 'lecun_uniform'}
0.7557 (0.0148) with: {'init_mode': 'normal'}
0.6515 (0.0020) with: {'init_mode': 'zeros'}
0.7345 (0.0100) with: {'init_mode': 'glorot_normal'}
0.7508 (0.0111) with: {'init_mode': 'glorot_uniform'}
0.7492 (0.0263) with: {'init_mode': 'he_normal'}
0.7410 (0.0217) with: {'init_mode': 'he_uniform'}


KerasWrapper(batch_size=60)

In [13]:
# Activation
res_act = run_grid(model, {'activation': ['softmax', 'softplus', 'softsign', 'relu',
                   'tanh', 'sigmoid', 'hard_sigmoid', 'linear']}, "Activation function")
best_act = res_act.best_params_['activation']
model.set_params(activation=best_act)

--- Activation function ---
Best: 0.7622 using {'activation': 'relu'}
0.6515 (0.0020) with: {'activation': 'softmax'}
0.7411 (0.0135) with: {'activation': 'softplus'}
0.7508 (0.0075) with: {'activation': 'softsign'}
0.7622 (0.0170) with: {'activation': 'relu'}
0.7573 (0.0089) with: {'activation': 'tanh'}
0.6710 (0.0147) with: {'activation': 'sigmoid'}
0.6628 (0.0077) with: {'activation': 'hard_sigmoid'}
0.7606 (0.0075) with: {'activation': 'linear'}


KerasWrapper(batch_size=60)

In [14]:
# Dropout
res_drop = run_grid(model, {'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]}, "Dropout rate")
best_drop = res_drop.best_params_['dropout_rate']
model.set_params(dropout_rate=best_drop)

--- Dropout rate ---
Best: 0.7638 using {'dropout_rate': 0.0}
0.7638 (0.0153) with: {'dropout_rate': 0.0}
0.7622 (0.0044) with: {'dropout_rate': 0.1}
0.7606 (0.0085) with: {'dropout_rate': 0.2}
0.7492 (0.0087) with: {'dropout_rate': 0.3}
0.7606 (0.0045) with: {'dropout_rate': 0.4}
0.7524 (0.0017) with: {'dropout_rate': 0.5}
0.7541 (0.0066) with: {'dropout_rate': 0.6}


KerasWrapper(batch_size=60)

In [15]:
# neurons

res_neu = run_grid(model, {'neurons': [1, 5, 10, 15, 20, 25, 30]}, "Number of neurons")
best_neurons = res_neu.best_params_['neurons']
model.set_params(neurons=best_neurons)

--- Number of neurons ---
Best: 0.7687 using {'neurons': 15}
0.6515 (0.0020) with: {'neurons': 1}
0.7638 (0.0153) with: {'neurons': 5}
0.7508 (0.0177) with: {'neurons': 10}
0.7687 (0.0085) with: {'neurons': 15}
0.7589 (0.0126) with: {'neurons': 20}
0.7557 (0.0187) with: {'neurons': 25}
0.7557 (0.0210) with: {'neurons': 30}


KerasWrapper(batch_size=60, neurons=15)